## Note
This script processes the movie features used for the encoding model, using *Despicable Me* (DM) as an example.  
The same procedure applies to *The Present* (TP).

Features for each movie include:

1. **Linguistic features (word level)**
   - GPT3_surprisal
   - Lg10WF (log10 word frequency)

2. **Nuisance regressors (10 Hz temporal resolution)** obtained from the EmoCodes dataset:
   https://wustl.app.box.com/v/emocodespublicdata/folder/165162908841

   - spoken_words  
   - written_words  
   - positive  
   - negative  
   - faces  
   - body  
   - brightness  
   - loudness

## a. Resample features using a Lanczos filter

The following code resamples the feature time series using a Lanczos filter.  
We use the *interpdata* function from the Huth Lab speech model tutorial:
https://github.com/HuthLab/speechmodeltutorial

Each original feature file contains two columns:
- onset: the onset time of each feature event
- a feature column named after the corresponding feature (e.g., GPT3_surprisal)

In [ ]:
import numpy as np
import pandas as pd
import os
from interpdata import lanczosinterp2D

nscans = 750  # TR for movie (250 for TP)
TR = 0.8  # TR in seconds TR=800ms
newtime = np.arange(0, (nscans * TR), TR)


input_folder = 'LanczosInterp_features/original/' #original features at word level or 10Hz level
output_folder = 'LanczosInterp_features/lanczosinterp/'

# List of contrasts/features
cons = ["GPT3_surprisal", "Lg10WF", "spoken_words", "written_words", 
        "positive", "negative", "faces", "body", "brightness", "loudness"]

for con in cons:
    df = pd.read_csv(os.path.join(input_folder, f"movieDM_{con}_ori.csv"))
    df[con] = df[con].fillna(0) #important!
    n_events = len(df)
    oldtime = df['onset'].values
    data = df[con].values

    downsampled_data = lanczosinterp2D(data, oldtime, newtime, window=3) # changed window!
    
    ds_df = pd.DataFrame(downsampled_data)
    ds_df.columns = [f'{con}_lanczos'] #add column name
    ds_df.to_csv(os.path.join(output_folder,f"movieDM_{con}_lanczos.csv"), index=False, header=True)
    print(f'Completed lanczos interpolation for {con} with shape {ds_df.shape}.')

## b. center features & construct feature matrix

In [ ]:
import os
import numpy as np
from sklearn.preprocessing import StandardScaler
import pandas as pd
import hdf5storage as hdf5

scaler = StandardScaler(with_mean=True, with_std=False)

input_folder = '/Users/xinyitang/Desktop/HBN_largesample/01_VoxelwiseEncodingModel/0_LanczosInterp_features/lanczosinterp/'
output_folder = 'LanczosInterp_features/lanczosinterp_centered/'
os.makedirs(output_folder, exist_ok=True)

# List of contrasts/features
cons = ["GPT3_surprisal", "Lg10WF", "spoken_words", "written_words", 
        "positive", "negative", "faces", "body", "brightness", "loudness"]

# Store centered feature arrays
feature_list = []

for con in cons:

    ds_df = pd.read_csv(os.path.join(input_folder,f"movieDM_{con}_lanczos.csv"))
    ds_data = ds_df[f'{con}_lanczos'].values
    ds_data = ds_data.reshape(-1, 1)
    
    ds_data_scaled = scaler.fit_transform(ds_data)
    
    # Append the data to the features list
    feature_list.append(ds_data_scaled)
    
    ds_data_scaled_df = pd.DataFrame(ds_data_scaled)
    ds_data_scaled_df.columns = [f'{con}_lanczos_cen'] #add column name
    ds_data_scaled_df.to_csv(os.path.join(output_folder,f"movieDM_{con}_lanczos_cen.csv"), index=False, header=True)
    print(f'Completed centering for {con} with shape {ds_df.shape}.')

# Convert the list of feature data into a NumPy array (DM shape: 10, 750)
features_matrix = np.column_stack(feature_list)
print("Final features matrix shape:", features_matrix.shape)  # Should be (750, 10)
features = {'features':features_matrix}

# save the features as a .hdf file
hdf5.writes(features, "movieDM_lexisurp_10features_lanczos.hdf",matlab_compatible=True)

Final features matrix shape: (750, 10)
Loaded features shape: (10, 750)
